In [5]:

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# This helper function is well-designed and does not need any changes.
def _build_quartile_lookup(original_correct_path,
                           original_misclassified_path,
                           significant_features_df):
    """
    Build per-stance quartile bins + (correct, misclass) rate look-ups.
    Returns: quartile_bins_lookup, correctness_rate_lookup, misclass_rate_lookup
    """
    print("Building quartile / correctness / misclass look-ups from training data…")

    df_cor = pd.read_csv(original_correct_path)
    df_mis = pd.read_csv(original_misclassified_path)
    df_cor["label"] = "Correct"
    df_mis["label"] = "Misclassified"
    df_all = pd.concat([df_cor, df_mis], ignore_index=True)

    quartile_bins_lookup  = {}   # feature ➜ stance ➜ bins
    correctness_rate_lp   = {}   # feature ➜ stance ➜ {interval: rate}
    misclass_rate_lp      = {}

    features = significant_features_df["feature"].unique()
    st_map   = {"All Stances": None, "AGAINST": 0, "FAVOR": 1, "NONE": 2}

    for feat in features:
        quartile_bins_lookup[feat] = {}
        correctness_rate_lp [feat] = {}
        misclass_rate_lp   [feat] = {}

        for st_name, st_val in st_map.items():
            sub = df_all if st_val is None else df_all[df_all["stance"] == st_val]
            if sub.empty or feat not in sub.columns or sub[feat].nunique() < 4:
                continue

            # stance-specific bins
            try:
                _, bins = pd.qcut(sub[feat], q=4, retbins=True, duplicates="drop")
            except ValueError:
                continue

            quartile_bins_lookup[feat][st_name] = bins
            sub = sub.assign(quantile=pd.cut(sub[feat], bins=bins, include_lowest=True))

            grp = sub.groupby(["quantile", "label"], observed=False).size().unstack(fill_value=0)
            grp["Total"]            = grp.sum(axis=1)
            grp["Correct Rate"]     = grp["Correct"]      / grp["Total"]
            grp["Misclassified Rate"] = grp["Misclassified"] / grp["Total"]

            correctness_rate_lp[feat][st_name] = grp["Correct Rate"].to_dict()
            misclass_rate_lp  [feat][st_name] = grp["Misclassified Rate"].to_dict()

    print("… look-ups ready.\n")
   
    return quartile_bins_lookup, correctness_rate_lp, misclass_rate_lp



# MODIFIED: The function now computes a single, relevant confidence score.
def evaluate_stylistic_confidence(evaluation_folder,
                                  significant_features_path,
                                  original_correct_path,
                                  original_misclassified_path,
                                  output_dir,
                                  coef_direction): # <-- NEW PARAMETER
    """
    Compute a single, per-sample stylistic confidence score based on the experiment type.
    """
    # 1. Load evaluation data
    print("[1] Loading evaluation set…")
    eval_df = pd.read_csv(os.path.join(evaluation_folder, "wtwt_test_processed.csv"))
    corr_idx = np.load(os.path.join(evaluation_folder, "correctly_classified_indices.npy"))
    mis_idx  = np.load(os.path.join(evaluation_folder, "misclassified_indices.npy"))

    eval_df["actual_outcome"] = "Correct"
    eval_df.loc[mis_idx, "actual_outcome"] = "Misclassified"
    
    # MODIFIED: Flipped labels to match the analysis script: Correct = 1.
    eval_df["actual_outcome_binary"] = eval_df["actual_outcome"].map(
        {"Correct": 1, "Misclassified": 0}).astype(int)

    # 2. Load the "ruleset" of significant features
    print(f"[2] Loading significant features for a '{coef_direction}' experiment.")
    sig_df = pd.read_csv(significant_features_path)
    
    # 3. Build look-ups from training data (this helper remains unchanged)
    bins_lp, corr_lp, mis_lp = _build_quartile_lookup(
        original_correct_path, original_misclassified_path, sig_df)

    # 4. Score each sample based on the experiment type
    print(f"[3] Calculating a single confidence score based on '{coef_direction}' features…")
    st_inv = {0: "AGAINST", 1: "FAVOR", 2: "NONE"}
    results = []

    for idx, row in eval_df.iterrows():
        st_name = st_inv.get(row.get("stance"))
        if st_name is None:
            continue

        feat_subset = sig_df[sig_df["stance"] == st_name]["feature"].unique()
        
        # MODIFIED: Collect rates for only the relevant score type.
        rates = []
        for feat in feat_subset:
            if pd.isna(row.get(feat)):
                continue
            
            bins = bins_lp.get(feat, {}).get(st_name)
            if bins is None:
                continue 
            
            interval = pd.cut([row[feat]], bins=bins, include_lowest=True)[0]

            # Get the rate corresponding to the experiment type
            if coef_direction == 'positive':
                rate = corr_lp.get(feat, {}).get(st_name, {}).get(interval)
            elif coef_direction == 'negative':
                rate = mis_lp.get(feat, {}).get(st_name, {}).get(interval)
            else:
                rate = None # Should not happen with your setup

            if rate is not None:
                rates.append(rate)

        # Calculate a single score for the sample
        score = np.mean(rates) if rates else np.nan
        
        results.append({
            "index": idx,
            "confidence_score": score,
            "actual_outcome": row["actual_outcome"]
        })

    results_df = pd.DataFrame(results).dropna()
    score_type = "Correctness" if coef_direction == 'positive' else "Misclassification"
    print("… scoring done.\n")

    # 5. Quick stats
    print("[4] Aggregate statistics")
    print(results_df.groupby("actual_outcome")[["confidence_score"]].mean().round(3))

    # 6. Plotting
    # MODIFIED: Generate a single boxplot with a dynamic title.
    plt.figure(figsize=(6, 5))
    sns.boxplot(x="actual_outcome", y="confidence_score",
                data=results_df, order=["Correct", "Misclassified"])
    plt.title(f"Expected Stylistic Confidence ({score_type})")
    plt.ylabel(f"Confidence Score ({score_type})")
    plt.xlabel("Actual Model Outcome")
    plt.tight_layout()

    out_plot_boxplot = os.path.join(output_dir, "confidence_boxplot.png")
    plt.savefig(out_plot_boxplot, dpi=300)
    print(f"Box plot saved to ➜ {out_plot_boxplot}")
    plt.close()

    # 7. Save detailed results
    out_csv = os.path.join(output_dir, "evaluation_results.csv")
    results_df.to_csv(out_csv, index=False)
    print(f"Evaluation results CSV saved to ➜ {out_csv}")
    
    return results_df

In [6]:
if __name__ == '__main__':
    sns.set_theme(style="whitegrid")

    # --- Define Experiments ---
    experiments_to_run = [
        {'coef': 'positive', 'sig': 'yes'},
        {'coef': 'negative', 'sig': 'yes'}
    ]
    coef_thresholds = np.round(np.arange(0.10, 1.00, 0.10), 2).tolist()
    
    for threshold in coef_thresholds:
        BASE_PLOTS_DIR = 'balanced_analysis'
        THRESHOLD_DIR = f"{threshold}_threshold_experiment"
        EVALUATION_DATA_FOLDER = 'evaluation_data'
        ORIGINAL_CORRECT_DATA_PATH = '/home/p/parush/style_markers/classifications/whole/correctly_classified_examples_processed.csv'
        ORIGINAL_MISCLASSIFIED_DATA_PATH = '/home/p/parush/style_markers/classifications/whole/misclassified_examples_processed.csv'
        dataset = 'whole'
        source = 'whole'

        # --- Loop Through and Run All Experiments ---
        for i, experiment_config in enumerate(experiments_to_run):
            coef_direction = experiment_config['coef']
            significance_filter = experiment_config['sig']

            print(f"\n{'='*60}")
            print(f"  RUNNING EVALUATION FOR EXPERIMENT {i+1}/{len(experiments_to_run)}")
            print(f"  Threshold: {threshold}, Coef Direction: '{coef_direction}', Sig Filter: '{significance_filter}'")
            print(f"{'='*60}\n")

            experiment_name = f"{source}_{dataset}_coef-{coef_direction}_sig-{significance_filter}"
            experiment_folder = os.path.join(BASE_PLOTS_DIR, THRESHOLD_DIR, experiment_name)
            ruleset_path = os.path.join(experiment_folder, "filtered_features_summary.csv")
            evaluation_output_dir = os.path.join(experiment_folder, "evaluation_results")
            os.makedirs(evaluation_output_dir, exist_ok=True)

            if not os.path.exists(ruleset_path):
                print(f"  [ERROR] Ruleset file not found, skipping: {ruleset_path}")
                continue

            # MODIFIED: Pass the coef_direction to the evaluation function.
            evaluation_results = evaluate_stylistic_confidence(
                evaluation_folder=EVALUATION_DATA_FOLDER,
                significant_features_path=ruleset_path,
                original_correct_path=ORIGINAL_CORRECT_DATA_PATH,
                original_misclassified_path=ORIGINAL_MISCLASSIFIED_DATA_PATH,
                output_dir=evaluation_output_dir,
                coef_direction=coef_direction # <-- Pass the parameter
            )

            # MODIFIED: Generate a single histogram for the 'confidence_score' column.
            print("\n[5] Generating additional plots...")
            score_type = "Correctness" if coef_direction == 'positive' else "Misclassification"
            
            plt.figure(figsize=(7, 4))
            sns.histplot(data=evaluation_results, x="confidence_score", hue="actual_outcome",
                         element="step", stat="density", common_norm=False, bins=20)
            plt.xlabel(f"Predicted Stylistic Confidence ({score_type})")
            plt.title("Histogram of Scores by True Outcome")
            plt.tight_layout()

            hist_plot_path = os.path.join(evaluation_output_dir, f"hist_confidence_score.png")
            plt.savefig(hist_plot_path, dpi=300)
            print(f"Histogram saved to ➜ {hist_plot_path}")
            plt.close()

    print("\n\nAll evaluations complete.")


  RUNNING EVALUATION FOR EXPERIMENT 1/2
  Threshold: 0.1, Coef Direction: 'positive', Sig Filter: 'yes'

[1] Loading evaluation set…
[2] Loading significant features for a 'positive' experiment.
Building quartile / correctness / misclass look-ups from training data…
… look-ups ready.

[3] Calculating a single confidence score based on 'positive' features…
… scoring done.

[4] Aggregate statistics
                confidence_score
actual_outcome                  
Correct                    0.777
Misclassified              0.812
Box plot saved to ➜ today_analysis/0.1_threshold_experiment/whole_whole_coef-positive_sig-yes/evaluation_results/confidence_boxplot.png
Evaluation results CSV saved to ➜ today_analysis/0.1_threshold_experiment/whole_whole_coef-positive_sig-yes/evaluation_results/evaluation_results.csv

[5] Generating additional plots...
Histogram saved to ➜ today_analysis/0.1_threshold_experiment/whole_whole_coef-positive_sig-yes/evaluation_results/hist_confidence_score.png

  RU